# 🔍 Pattern Hunters: Enzyme Inhibition
### A Detective Notebook — From Anomalous Data to Molecular Diagnosis
**Kuchinda Degree College | Department of Zoology | Sambalpur University**

---
> *"In a real experiment you do not know the inhibition type in advance.*  
> *You see something wrong. You ask why. The data answers — if you know how to read it."*

---
## How to use this notebook

This is **Notebook 2** in the Pattern Hunters Biochemistry series.  
**Notebook 1** (*How the MM Equation Emerges from Nature*) should be completed first.

This notebook is structured as a **detective story**.  
Each section begins with anomalous data — a rate that does not behave as expected.  
Your job is to reason from the pattern to the mechanism.

The four inhibition types are not categories to memorise.  
They are **four different physical situations** — each with a different molecular story,  
a different experimental signature, and a different biological consequence.

---
### Notebook 2 contents
1. Something is wrong — the concept of an inhibitor  
2. Where does the inhibitor bind? — four physical situations  
3. Competitive inhibition — substrate and inhibitor fight for the same site  
4. Non-competitive inhibition — the inhibitor is indifferent to substrate  
5. Uncompetitive inhibition — the inhibitor waits for substrate to arrive  
6. Mixed inhibition — the general case  
7. The Lineweaver-Burk as a diagnostic tool  
8. Reversible vs irreversible — heavy metals as a special case  
9. IC₅₀ and the dose-response curve  
10. 🌍 Western Odisha — heavy metals as soil enzyme inhibitors  
11. 🧪 Your own inhibition experiment  
Summary — the complete inhibition detective toolkit


In [ ]:
# ============================================================
# RUN THIS FIRST — imports and all simulation functions
# ============================================================
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.optimize import curve_fit
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor':'white','axes.facecolor':'#fafafa',
    'axes.spines.top':False,'axes.spines.right':False,
    'axes.grid':True,'grid.alpha':0.3,'grid.linestyle':'--',
    'font.size':12,'axes.labelsize':12,'axes.titlesize':13,
    'lines.linewidth':2.2,
})

# ── Core kinetic functions ─────────────────────────────────
def v_noinhib(S, Vmax, Km):
    return (Vmax * S) / (Km + S)

def v_competitive(S, Vmax, Km, I, Ki):
    """Competitive: inhibitor raises apparent Km. Vmax unchanged."""
    Km_app = Km * (1 + I / Ki)
    return (Vmax * S) / (Km_app + S)

def v_noncompetitive(S, Vmax, Km, I, Ki):
    """Non-competitive: inhibitor lowers apparent Vmax. Km unchanged."""
    Vmax_app = Vmax / (1 + I / Ki)
    return (Vmax_app * S) / (Km + S)

def v_uncompetitive(S, Vmax, Km, I, Ki):
    """Uncompetitive: both Vmax and Km fall by same factor."""
    alpha_p = 1 + I / Ki
    Vmax_app = Vmax / alpha_p
    Km_app   = Km   / alpha_p
    return (Vmax_app * S) / (Km_app + S)

def v_mixed(S, Vmax, Km, I, Ki, Ki_prime):
    """Mixed: inhibitor binds E (Ki) and ES (Ki_prime) with different affinities."""
    alpha  = 1 + I / Ki
    alpha_p= 1 + I / Ki_prime
    return (Vmax * S) / (alpha * Km + alpha_p * S)

def simulate_progress(S0, Vmax, Km, t_max=60, n=500,
                       inh_type=None, I=0, Ki=50, Ki_prime=50):
    dt = t_max / n
    times = np.linspace(0, t_max, n+1)
    S, P = S0, 0.0
    S_arr, P_arr = [S0], [0.0]
    for _ in range(n):
        if   inh_type == 'competitive':    v = v_competitive(S, Vmax, Km, I, Ki)
        elif inh_type == 'noncompetitive': v = v_noncompetitive(S, Vmax, Km, I, Ki)
        elif inh_type == 'uncompetitive':  v = v_uncompetitive(S, Vmax, Km, I, Ki)
        elif inh_type == 'mixed':          v = v_mixed(S, Vmax, Km, I, Ki, Ki_prime)
        else:                               v = v_noinhib(S, Vmax, Km)
        dP = max(0, v) * dt
        P = min(P + dP, S0); S = max(0, S0 - P)
        S_arr.append(S); P_arr.append(P)
    return times, np.array(S_arr), np.array(P_arr)

# ── Global parameters ──────────────────────────────────────
Vmax, Km = 80.0, 25.0   # baseline enzyme
I_std, Ki_std = 30.0, 20.0   # standard inhibitor scenario

print('Setup complete. You are now a Pattern Hunter investigating anomalous enzyme behaviour.')
print(f'Baseline enzyme: Vmax = {Vmax} umol/min,  Km = {Km} mM')


---
## Section 1 — Something is Wrong
### The concept of an inhibitor — starting from anomalous data

You have characterised your enzyme carefully. You know its Km and Vmax.  
You run the assay again — same enzyme, same substrate concentration.  
**The rate is lower than expected.**

This is not measurement error. You have repeated it five times.  
Something in the reaction mixture is slowing the enzyme down.

**Before running the cell — write your hypotheses in your notebook:**  
*List three different molecular reasons why an enzyme's rate might be lower than expected.*


In [ ]:
# === SECTION 1: Observing the anomaly ===

S_range = np.linspace(0.1, 200, 400)
I_concentrations = [0, 10, 20, 40, 80]  # uM inhibitor
colors_inh = ['#1D9E75','#7F77DD','#BA7517','#D85A30','#E24B4A']

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Section 1 — Something is Wrong: Rate is Lower Than Expected',
             fontsize=13, fontweight='bold')

# Left: MM curves at increasing inhibitor concentration
ax1 = axes[0]
for I, col in zip(I_concentrations, colors_inh):
    v = v_competitive(S_range, Vmax, Km, I, Ki_std)
    lbl = 'No inhibitor' if I == 0 else f'[I] = {I} uM'
    ax1.plot(S_range, v, color=col, label=lbl)

ax1.axhline(Vmax, color='#BA7517', linestyle='--', alpha=0.4, label=f'Expected Vmax={Vmax}')
ax1.set_xlabel('[S]  (mM)'); ax1.set_ylabel('v0  (umol/min)')
ax1.set_title('Rate falls as inhibitor concentration rises\n(type unknown at this stage)', fontsize=11)
ax1.legend(fontsize=9)

# Right: Progress curve comparison — with and without inhibitor
ax2 = axes[1]
for I, col in zip([0, 40, 80], ['#1D9E75','#D85A30','#E24B4A']):
    t, _, P = simulate_progress(100, Vmax, Km, inh_type='competitive', I=I, Ki=Ki_std)
    lbl = 'No inhibitor' if I==0 else f'[I] = {I} uM'
    ax2.plot(t, P, color=col, label=lbl)

ax2.set_xlabel('Time (min)'); ax2.set_ylabel('[P] formed (mM)')
ax2.set_title('Progress curves: inhibitor slows product formation\n[S]0 = 100 mM in all cases', fontsize=11)
ax2.legend(fontsize=9)

plt.tight_layout()
plt.savefig('inh_s1_anomaly.png', dpi=150, bbox_inches='tight')
plt.show()

print('DETECTIVE QUESTIONS — answer before moving on:')
print('  1. The rate is lower. Is the enzyme damaged, or is something competing with it?')
print('     How would you design an experiment to distinguish these two possibilities?')
print('  2. If you added MORE substrate, would the rate recover?')
print('     What would it mean if it did? What would it mean if it did not?')
print('  3. The inhibitor concentration varies from 0 to 80 uM.')
print('     Does the SHAPE of the curve change, or just its position?')
print('     What does the shape tell you that the position cannot?')


---
## Section 2 — Where Does the Inhibitor Bind?
### Four physical situations — not four named categories

The key insight of inhibition biochemistry is this:  
**the type of inhibition is determined entirely by where the inhibitor binds.**

There are only three places it can bind:

| Binding site | What is there | Consequence |
|-------------|--------------|-------------|
| Active site of free enzyme (E) | Substrate binding pocket | Competes directly with substrate |
| Allosteric site of free enzyme (E) | Regulatory site, not active site | Changes enzyme shape regardless of substrate |
| Allosteric site of ES complex only | Only accessible after substrate binds | Cannot bind free enzyme at all |

These three binding possibilities give rise to four inhibition types:  
competitive (active site of E only), non-competitive (allosteric site of E only),  
uncompetitive (allosteric site of ES only), and mixed (allosteric site of both E and ES).

**Before running the cell — draw the four binding scenarios in your notebook.**  
Draw E, S, ES, and I. Show which complex I binds in each case.


In [ ]:
# === SECTION 2: Four binding locations — molecular diagrams ===

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
fig.suptitle('Section 2 — Where the Inhibitor Binds Determines Everything',
             fontsize=13, fontweight='bold')

scenarios = [
    dict(title='Competitive', color='#7F77DD',
         binds='Active site of FREE enzyme (E)',
         mechanism='I mimics substrate shape. Only one occupant\npossible: either S or I, never both.',
         consequence='Adding more S displaces I.\nVmax preserved at saturating [S].\nKm rises (apparent).'),
    dict(title='Non-competitive', color='#D85A30',
         binds='Allosteric site of FREE enzyme (E)',
         mechanism='I binds a separate site. Does not affect\nsubstrate binding. Distorts active site shape.',
         consequence='More substrate cannot help.\nVmax falls. Km unchanged.'),
    dict(title='Uncompetitive', color='#BA7517',
         binds='Allosteric site of ES complex only',
         mechanism='I cannot bind free E. It waits for S to\narrive and reshape the enzyme, exposing\nits binding site.',
         consequence='More substrate makes it WORSE (more ES = more I binding).\nBoth Vmax and Km fall equally.'),
    dict(title='Mixed', color='#E24B4A',
         binds='Allosteric site of BOTH E and ES\n(with different affinities)',
         mechanism='General case. I binds free E (Ki)\nand ES complex (Ki-prime) but prefers one.',
         consequence='Vmax always falls.\nKm may rise or fall\ndepending on Ki vs Ki-prime.'),
]

for idx, (ax, sc) in enumerate(zip(axes.flatten(), scenarios)):
    ax.set_xlim(0, 10); ax.set_ylim(0, 8); ax.axis('off')
    ax.set_title(sc['title'], fontsize=13, fontweight='bold', color=sc['color'])

    # Draw enzyme (E)
    e_rect = plt.Rectangle((0.5, 4.5), 3.2, 2.5, color=sc['color'],
                             alpha=0.18, ec=sc['color'], lw=1.5)
    ax.add_patch(e_rect)
    ax.text(2.1, 6.2, 'E', ha='center', fontsize=14, fontweight='bold', color=sc['color'])

    # Active site pocket
    pocket = plt.Rectangle((1.0, 4.5), 1.4, 1.2, color=sc['color'],
                             alpha=0.4, ec=sc['color'], lw=1)
    ax.add_patch(pocket)
    ax.text(1.7, 4.9, 'active\nsite', ha='center', fontsize=7, color=sc['color'])

    # Allosteric site (shown for non-competitive, uncompetitive, mixed)
    if idx != 0:
        allo = plt.Rectangle((2.3, 4.5), 1.1, 1.0, color='#888780',
                               alpha=0.3, ec='#888780', lw=1)
        ax.add_patch(allo)
        ax.text(2.85, 4.88, 'allosteric\nsite', ha='center', fontsize=6.5, color='#555')

    # Substrate S
    s_circ = plt.Circle((6.0, 5.8), 0.6, color='#1D9E75', alpha=0.8)
    ax.add_patch(s_circ)
    ax.text(6.0, 5.8, 'S', ha='center', va='center', fontsize=11,
             fontweight='bold', color='white')

    # Inhibitor I — position depends on type
    if idx == 0:   ix, iy = 6.0, 3.2   # near active site
    elif idx == 1: ix, iy = 6.0, 3.2   # near allosteric of E
    elif idx == 2: ix, iy = 6.0, 3.2   # near ES allosteric
    else:          ix, iy = 6.0, 3.2

    i_rect = plt.Rectangle((ix-0.55, iy-0.45), 1.1, 0.9,
                             color=sc['color'], alpha=0.8,
                             ec=sc['color'], lw=1.5)
    ax.add_patch(i_rect)
    ax.text(ix, iy, 'I', ha='center', va='center', fontsize=11,
             fontweight='bold', color='white')

    # Binding site label
    ax.text(5.0, 7.5, f'Binds: {sc["binds"]}',
             fontsize=7.5, color=sc['color'], ha='center',
             bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                        edgecolor=sc['color'], alpha=0.9))

    # Mechanism and consequence text
    ax.text(0.3, 3.6, f'Mechanism:\n{sc["mechanism"]}',
             fontsize=8, color='#444', va='top')
    ax.text(0.3, 1.2, f'Effect:\n{sc["consequence"]}',
             fontsize=8.5, color=sc['color'], va='top', fontweight='bold')

plt.tight_layout()
plt.savefig('inh_s2_binding_sites.png', dpi=150, bbox_inches='tight')
plt.show()

print('KEY QUESTION before moving to Section 3:')
print('  For each type, predict: if you double [S], what happens to the rate?')
print('  Competitive:    rate will ________ because ________')
print('  Non-competitive: rate will ________ because ________')
print('  Uncompetitive:  rate will ________ because ________')
print('  Mixed:          rate will ________ because ________')


---
## Section 3 — Competitive Inhibition
### Substrate and inhibitor fight for the same site

> **BSc entry point:** Focus on the physical picture and the two observable consequences — Km rises, Vmax stays.
> Run the cell, observe the plots, answer the questions at the bottom.

> **MSc extension:** Work through the derivation block below before running the cell.
> Derive Km_app from the steady-state assumption. Then verify your algebra against the plot.

---
### The molecular story

The inhibitor structurally resembles the substrate. It fits into the active site — but unlike  
substrate, it is not converted to product. It just sits there, occupying the site.

At any moment, each enzyme molecule has exactly one occupant: S, or I, or neither. Never both.

**The critical question before running the cell:**  
*If you gradually increase [S] while holding [I] constant, what happens to the rate?*  
*At very high [S], nearly every encounter with the active site is a substrate molecule.*  
*The inhibitor is statistically outcompeted. Can the enzyme still reach Vmax?*

---
### BSc: Observable consequences

| Parameter | Effect | Why |
|-----------|--------|-----|
| Km | **Rises** (apparent Km = Km × α) | Enzyme appears to have lower affinity — needs more S to half-saturate |
| Vmax | **Unchanged** | At saturating [S], all sites occupied by S — inhibitor fully outcompeted |
| LB plot | Lines meet at **Y-axis** | Same y-intercept (1/Vmax), different slopes |

### MSc: The derivation

Starting from the modified mechanism: E + I ⇌ EI (non-productive)

At steady state, include EI in enzyme conservation:  
```
[E]total = [E] + [ES] + [EI]

[EI] = [E][I]/Ki        (from equilibrium Ki = [E][I]/[EI])

Substituting into [ES] = [E][S]/Km and solving:

  v = Vmax * [S] / (Km*alpha + [S])    where alpha = (1 + [I]/Ki)

     = Vmax * [S] / (Km_app + [S])     where Km_app = Km * (1 + [I]/Ki)
```
Note: Vmax does not contain alpha. It cancels out because at saturating [S], [EI] → 0.

**MSc question:** At what [S] does v_inhibited = v_uninhibited / 2?  
Show algebraically that this occurs at [S] = Km_app, not Km.

**MSc question:** The Lineweaver-Burk form is: 1/v = (Km_app/Vmax)(1/[S]) + 1/Vmax  
The slope = Km_app/Vmax = (Km/Vmax)(1 + [I]/Ki).  
Plot slope vs [I] — the x-intercept gives −Ki directly. Derive why.


In [ ]:
# === SECTION 3: Competitive Inhibition — Full Depth ===
# BSc: run and observe panels A, B, C
# MSc: additionally work through panels D and E (Ki extraction + algebra verification)

I_vals  = [0, 10, 20, 40, 80]
cols    = ['#1D9E75','#9FE1CB','#7F77DD','#D85A30','#E24B4A']
S_range = np.linspace(0.1, 250, 400)
S_lb    = np.array([5, 8, 12, 20, 35, 60, 100, 160])

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Section 3 — Competitive Inhibition: From Molecular Story to Ki Extraction',
             fontsize=13, fontweight='bold')

# ── Panel A: v vs [S] ──────────────────────────────────────────
ax1 = axes[0,0]
for I, col in zip(I_vals, cols):
    v = v_competitive(S_range, Vmax, Km, I, Ki_std)
    Km_a = Km*(1+I/Ki_std)
    ax1.plot(S_range, v, color=col,
              label=f'[I]={I}  Km_app={Km_a:.0f}')
ax1.axhline(Vmax, color='#BA7517', linestyle='--', alpha=0.5,
             label=f'Vmax={Vmax} (unchanged)')
ax1.set_xlabel('[S]  (mM)'); ax1.set_ylabel('v0  (umol/min)')
ax1.set_title('BSc Panel A: All curves share Vmax\nKm_app rises with [I]', fontsize=10)
ax1.legend(fontsize=7.5)

# ── Panel B: Lineweaver-Burk ───────────────────────────────────
ax2 = axes[0,1]
slopes_lb = []
for I, col in zip(I_vals, cols):
    v_lb = v_competitive(S_lb, Vmax, Km, I, Ki_std)
    inv_s, inv_v = 1/S_lb, 1/v_lb
    ax2.plot(inv_s, inv_v, 'o-', color=col, markersize=4,
              label=f'[I]={I}')
    # Fit line for slope extraction
    coeffs = np.polyfit(inv_s, inv_v, 1)
    slopes_lb.append(coeffs[0])
    # Extend line to y-axis
    x_ext = np.linspace(-0.02, max(inv_s), 100)
    ax2.plot(x_ext, np.polyval(coeffs, x_ext), color=col,
              alpha=0.3, linewidth=1)

ax2.axvline(0, color='#aaa', lw=0.8)
ax2.axhline(0, color='#aaa', lw=0.8)
ax2.axhline(1/Vmax, color='#BA7517', linestyle=':', alpha=0.7,
             linewidth=1.5, label=f'y-int = 1/Vmax = {1/Vmax:.4f}')
ax2.set_xlim(-0.025, 0.22)
ax2.set_xlabel('1/[S]  (mM⁻¹)'); ax2.set_ylabel('1/v0  (min/umol)')
ax2.set_title('BSc Panel B: Lines meet at Y-axis\n(identical y-intercept = 1/Vmax)', fontsize=10)
ax2.legend(fontsize=7.5)
ax2.annotate('All lines converge\non Y-axis (1/Vmax)',
              xy=(0, 1/Vmax), xytext=(0.04, 1/Vmax+0.006),
              arrowprops=dict(arrowstyle='->', color='#085041'),
              fontsize=8, color='#085041')

# ── Panel C: What happens at very high [S]? BSc intuition panel ──
ax3 = axes[0,2]
S_high = np.linspace(0.1, 2000, 800)
for I, col in zip([0, 40, 80], ['#1D9E75','#D85A30','#E24B4A']):
    v = v_competitive(S_high, Vmax, Km, I, Ki_std)
    ax3.semilogx(S_high, v, color=col, label=f'[I]={I} uM')
ax3.axhline(Vmax, color='#BA7517', linestyle='--', alpha=0.6,
             label=f'Vmax={Vmax} (all converge here)')
ax3.set_xlabel('[S]  (mM, log scale)'); ax3.set_ylabel('v0  (umol/min)')
ax3.set_title('BSc Panel C: Extend [S] to very high values\nAll curves converge on Vmax', fontsize=10)
ax3.legend(fontsize=9)
ax3.text(0.05, 0.15,
          'At [S] >> Km_app,\nsubstrate outcompetes\ninhibitor completely',
          transform=ax3.transAxes, fontsize=8.5, color='#085041',
          bbox=dict(boxstyle='round', facecolor='#E1F5EE', edgecolor='#1D9E75'))

# ── Panel D: MSc — slope vs [I] to extract Ki ────────────────────
ax4 = axes[1,0]
ax4.scatter(I_vals, slopes_lb, color='#7F77DD', s=80, zorder=5,
             label='LB slope at each [I]')
# Fit line through slope vs [I]
coeffs_s = np.polyfit(I_vals, slopes_lb, 1)
I_fit = np.linspace(-Ki_std*1.2, max(I_vals)*1.1, 200)
ax4.plot(I_fit, np.polyval(coeffs_s, I_fit), color='#7F77DD',
          linewidth=2, label='Linear fit')
ax4.axhline(0, color='#aaa', lw=0.8)
ax4.axvline(0, color='#aaa', lw=0.8)
Ki_extracted = -coeffs_s[1]/coeffs_s[0]
ax4.scatter([Ki_extracted], [0], color='#E24B4A', s=120, zorder=6,
             label=f'x-int = -Ki = {Ki_extracted:.1f} uM')
ax4.annotate(f'-Ki = {Ki_extracted:.1f} uM\n(True Ki = {Ki_std})',
              xy=(Ki_extracted, 0), xytext=(Ki_extracted+5, slopes_lb[1]*0.5),
              arrowprops=dict(arrowstyle='->', color='#E24B4A'),
              fontsize=8.5, color='#E24B4A')
ax4.set_xlabel('[I]  (uM)'); ax4.set_ylabel('LB slope  (= Km_app/Vmax)')
ax4.set_title('MSc Panel D: Extract Ki from slope vs [I]\nSlope = (Km/Vmax)(1 + [I]/Ki)', fontsize=10)
ax4.legend(fontsize=8)

# ── Panel E: MSc — Km_app vs [I], Dixon plot approach ─────────
ax5 = axes[1,1]
I_cont = np.linspace(0, 100, 200)
Km_app_cont = Km*(1 + I_cont/Ki_std)
ax5.plot(I_cont, Km_app_cont, color='#7F77DD', linewidth=2.5,
          label=f'Km_app = Km(1+[I]/Ki)')
# Extrapolate to Km_app = 0 (not physically meaningful but shows Ki)
I_extrap = np.linspace(-Ki_std*1.3, 100, 200)
Km_extrap = Km*(1 + I_extrap/Ki_std)
ax5.plot(I_extrap, Km_extrap, color='#7F77DD', linewidth=1.5,
          linestyle='--', alpha=0.5)
ax5.axvline(-Ki_std, color='#E24B4A', linestyle=':', alpha=0.8)
ax5.axhline(Km, color='#1D9E75', linestyle='--', alpha=0.5,
             label=f'True Km = {Km} mM')
ax5.scatter([0], [Km], color='#1D9E75', s=100, zorder=5)
ax5.scatter(I_vals, [Km*(1+I/Ki_std) for I in I_vals],
             color='#7F77DD', s=70, zorder=5)
ax5.annotate(f'x-intercept = -Ki = -{Ki_std}',
              xy=(-Ki_std, 0), xytext=(-Ki_std+8, Km*0.6),
              arrowprops=dict(arrowstyle='->', color='#E24B4A'),
              fontsize=8.5, color='#E24B4A')
ax5.set_xlim(-35, 110); ax5.set_ylim(-5, 90)
ax5.set_xlabel('[I]  (uM)'); ax5.set_ylabel('Km_app  (mM)')
ax5.set_title('MSc Panel E: Km_app rises linearly with [I]\nx-intercept = -Ki (Dixon-style)', fontsize=10)
ax5.legend(fontsize=9)

# ── Panel F: MSc — verify Cheng-Prusoff IC50 prediction ──────
ax6 = axes[1,2]
S_test_vals = [Km/4, Km/2, Km, Km*2, Km*4]
I_dr = np.logspace(-1, 3, 300)
cols_s = plt.cm.Greens(np.linspace(0.35, 0.9, len(S_test_vals)))
IC50_measured = []
for S_t, col in zip(S_test_vals, cols_s):
    acts = [v_competitive(S_t, Vmax, Km, I, Ki_std) /
             v_noinhib(S_t, Vmax, Km) * 100
             for I in I_dr]
    ax6.semilogx(I_dr, acts, color=col,
                  label=f'[S]={S_t:.1f} mM')
    # Find IC50 by interpolation
    acts_arr = np.array(acts)
    idx = np.argmin(np.abs(acts_arr - 50))
    IC50_measured.append(I_dr[idx])

IC50_theory = [Ki_std*(1+S/Km) for S in S_test_vals]
ax6.axhline(50, color='#E24B4A', linestyle='--', alpha=0.7)
ax6.set_xlabel('[I]  (uM, log)'); ax6.set_ylabel('% activity')
ax6.set_title('MSc Panel F: IC50 rises with [S]\n(Cheng-Prusoff: IC50 = Ki*(1+[S]/Km))', fontsize=10)
ax6.legend(fontsize=7.5)

plt.tight_layout()
plt.savefig('inh_s3_competitive_deep.png', dpi=150, bbox_inches='tight')
plt.show()

print('COMPETITIVE INHIBITION — PARAMETER SUMMARY:')
print(f'  True Ki = {Ki_std} uM')
print(f'  Ki extracted from LB slope plot = {Ki_extracted:.1f} uM')
print()
print('BSc CHECK (answer in notebook):')
print('  1. Do all LB lines share the same y-intercept? Read it from Panel B.')
print('     y-intercept = ____   therefore Vmax = ____')
print('  2. At [I]=40 uM, Km_app = ____. How much more substrate is needed')
print('     to reach half-maximal rate compared to no inhibitor?')
print()
print('MSc EXTENSION (derive, then verify):')
print('  1. From Panel D: slope = (Km/Vmax) + (Km/Vmax/Ki)*[I]')
print(f'     Intercept should be Km/Vmax = {Km/Vmax:.4f}')
print(f'     Measured from fit: {coeffs_s[1]:.4f}   <- should match')
print(f'     Slope of slope-vs-[I] should be Km/(Vmax*Ki) = {Km/(Vmax*Ki_std):.5f}')
print(f'     Measured: {coeffs_s[0]:.5f}   <- should match')
print()
print('  2. Cheng-Prusoff verification:')
for S_t, IC50_m, IC50_t in zip(S_test_vals, IC50_measured, IC50_theory):
    print(f'     [S]={S_t:5.1f} mM: IC50 measured={IC50_m:.1f} uM, '
          f'theory={IC50_t:.1f} uM  (Ki*(1+{S_t:.1f}/{Km}))')


---
## Section 4 — Non-competitive Inhibition
### The inhibitor is indifferent to substrate

> **BSc entry point:** Focus on the key observable difference from competitive —
> Vmax falls, Km unchanged. Run the cell and compare Panel C directly.

> **MSc extension:** Derive why Km is unchanged. What does the LB y-intercept
> vs [I] plot tell you? Work through the Dixon plot in Panel E to extract Ki.

---
### The molecular story

The inhibitor binds a **separate allosteric site** — not the active site.  
It binds free enzyme (E) and ES complex with equal affinity.  
It distorts the enzyme's shape, reducing kcat (the catalytic step) —  
but it does not affect how substrate enters or leaves the active site.

**The critical question before running the cell:**  
*The inhibitor has nothing to do with substrate binding.*  
*If you add more substrate, does the inhibitor get displaced?*  
*Think about this before predicting whether Vmax changes.*

---
### BSc: Observable consequences

| Parameter | Effect | Why |
|-----------|--------|-----|
| Km | **Unchanged** | Substrate still binds just as well — inhibitor is elsewhere |
| Vmax | **Falls** (Vmax_app = Vmax / (1 + [I]/Ki)) | Every enzyme molecule works slower |
| LB plot | Lines meet at **X-axis** | Same x-intercept (−1/Km), different y-intercepts |

### MSc: The derivation

The inhibitor creates two new species: EI and ESI.
```
[E]total = [E] + [ES] + [EI] + [ESI]

Since I binds E and ES with EQUAL affinity Ki:
  [EI]  = [E][I]/Ki
  [ESI] = [ES][I]/Ki

Fraction of enzyme as productive ES:
  [ES] / [E]total = [S]/Km / ((1 + [I]/Ki)(1 + [S]/Km))

Therefore:
  v = Vmax * [S] / (Km * alpha + [S] * alpha)    where alpha = 1 + [I]/Ki
    = (Vmax/alpha) * [S] / (Km + [S])
    = Vmax_app * [S] / (Km + [S])
```
Km does not change because alpha divides both numerator terms equally.

**MSc question:** Why does alpha appear in BOTH the numerator and denominator?  
For competitive inhibition, alpha only appeared in the Km term. What physical  
assumption is different here that causes it to appear in both?

**MSc question:** The LB form is: 1/v = (Km/Vmax)(1/[S]) + (1/Vmax)*alpha  
The slope = Km/Vmax (unchanged with [I]).  
The y-intercept = alpha/Vmax (rises with [I]).  
Plot y-intercept vs [I] — what is the x-intercept? Derive why it equals -Ki.


In [ ]:
# === SECTION 4: Non-competitive Inhibition — Full Depth ===
# BSc: run and observe panels A, B, C
# MSc: additionally work through panels D, E, F

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Section 4 — Non-competitive Inhibition: Substrate Cannot Help',
             fontsize=13, fontweight='bold')

# ── Panel A: v vs [S] ──────────────────────────────────────
ax1 = axes[0,0]
for I, col in zip(I_vals, cols):
    v = v_noncompetitive(S_range, Vmax, Km, I, Ki_std)
    Vmax_a = Vmax/(1+I/Ki_std)
    ax1.plot(S_range, v, color=col,
              label=f'[I]={I}  Vmax_app={Vmax_a:.0f}')
ax1.axvline(Km, color='#1D9E75', linestyle=':', alpha=0.5,
             label=f'Km={Km} (unchanged)')
ax1.set_xlabel('[S]  (mM)'); ax1.set_ylabel('v0  (umol/min)')
ax1.set_title('BSc Panel A: Curves reach DIFFERENT Vmax\nKm unchanged — inflection at same [S]', fontsize=10)
ax1.legend(fontsize=7.5)

# ── Panel B: Lineweaver-Burk ───────────────────────────────
ax2 = axes[0,1]
y_ints_lb = []
for I, col in zip(I_vals, cols):
    v_lb = v_noncompetitive(S_lb, Vmax, Km, I, Ki_std)
    inv_s, inv_v = 1/S_lb, 1/v_lb
    ax2.plot(inv_s, inv_v, 'o-', color=col, markersize=4, label=f'[I]={I}')
    coeffs = np.polyfit(inv_s, inv_v, 1)
    y_ints_lb.append(coeffs[1])
    x_ext = np.linspace(-1/Km*1.1, max(inv_s), 100)
    ax2.plot(x_ext, np.polyval(coeffs, x_ext), color=col, alpha=0.3, linewidth=1)

ax2.axvline(-1/Km, color='#1D9E75', linestyle=':', alpha=0.7,
             label=f'x-int = -1/Km = {-1/Km:.3f}')
ax2.axvline(0, color='#aaa', lw=0.8); ax2.axhline(0, color='#aaa', lw=0.8)
ax2.set_xlim(-0.06, 0.22)
ax2.set_xlabel('1/[S]  (mM⁻¹)'); ax2.set_ylabel('1/v0  (min/umol)')
ax2.set_title('BSc Panel B: Lines meet at X-axis\n(identical x-intercept = −1/Km)', fontsize=10)
ax2.legend(fontsize=7.5)
ax2.annotate('All lines converge\non X-axis (-1/Km)',
              xy=(-1/Km, 0), xytext=(-1/Km+0.03, 0.01),
              arrowprops=dict(arrowstyle='->', color='#085041'),
              fontsize=8, color='#085041')

# ── Panel C: Direct comparison with competitive ────────────────
ax3 = axes[0,2]
I_d = 40
v_none  = v_noinhib(S_range, Vmax, Km)
v_comp  = v_competitive(S_range, Vmax, Km, I_d, Ki_std)
v_nonc  = v_noncompetitive(S_range, Vmax, Km, I_d, Ki_std)
ax3.plot(S_range, v_none, color='#1D9E75', lw=2.5, label='No inhibitor')
ax3.plot(S_range, v_comp, color='#7F77DD', lw=2,   linestyle='--', label=f'Competitive [I]={I_d}')
ax3.plot(S_range, v_nonc, color='#E24B4A', lw=2,   linestyle=':',  label=f'Non-competitive [I]={I_d}')
# Annotate Km position for both
ax3.axvline(Km,                    color='#1D9E75', linestyle=':', alpha=0.4, lw=1.2)
ax3.axvline(Km*(1+I_d/Ki_std),     color='#7F77DD', linestyle=':', alpha=0.4, lw=1.2)
ax3.annotate(f'Comp: Km_app={Km*(1+I_d/Ki_std):.0f}',
              xy=(Km*(1+I_d/Ki_std), Vmax/4),
              xytext=(Km*(1+I_d/Ki_std)+15, Vmax/4+8),
              fontsize=8, color='#7F77DD',
              arrowprops=dict(arrowstyle='->', color='#7F77DD'))
ax3.annotate(f'Non-comp: Km={Km} (same)',
              xy=(Km, v_nonc[np.argmin(np.abs(S_range-Km))]),
              xytext=(Km+25, v_nonc[np.argmin(np.abs(S_range-Km))]-10),
              fontsize=8, color='#E24B4A',
              arrowprops=dict(arrowstyle='->', color='#E24B4A'))
ax3.set_xlabel('[S] (mM)'); ax3.set_ylabel('v0 (umol/min)')
ax3.set_title('BSc Panel C: Competitive vs non-competitive\nSame [I], different molecular stories', fontsize=10)
ax3.legend(fontsize=8)

# ── Panel D: MSc — y-intercept vs [I] to extract Ki ──────────
ax4 = axes[1,0]
ax4.scatter(I_vals, y_ints_lb, color='#D85A30', s=80, zorder=5,
             label='LB y-intercept at each [I]')
coeffs_y = np.polyfit(I_vals, y_ints_lb, 1)
I_fit = np.linspace(-Ki_std*1.3, max(I_vals)*1.1, 200)
ax4.plot(I_fit, np.polyval(coeffs_y, I_fit), color='#D85A30', lw=2, label='Linear fit')
ax4.axhline(0, color='#aaa', lw=0.8); ax4.axvline(0, color='#aaa', lw=0.8)
Ki_nc = -coeffs_y[1]/coeffs_y[0]
ax4.scatter([Ki_nc], [0], color='#E24B4A', s=120, zorder=6,
             label=f'x-int = -Ki = {Ki_nc:.1f} uM')
ax4.annotate(f'-Ki = {Ki_nc:.1f} uM\n(True Ki = {Ki_std})',
              xy=(Ki_nc, 0), xytext=(Ki_nc+5, coeffs_y[1]*0.5),
              arrowprops=dict(arrowstyle='->', color='#E24B4A'),
              fontsize=8.5, color='#E24B4A')
ax4.set_xlabel('[I]  (uM)'); ax4.set_ylabel('LB y-intercept (= 1/Vmax_app)')
ax4.set_title('MSc Panel D: y-intercept rises with [I]\nx-intercept = -Ki', fontsize=10)
ax4.legend(fontsize=8)

# ── Panel E: MSc — Dixon plot (1/v vs [I] at fixed [S]) ─────
ax5 = axes[1,1]
I_dixon = np.linspace(0, 100, 200)
S_dixon_vals = [10, 25, 50, 100]
cols_d = plt.cm.Purples(np.linspace(0.35, 0.9, len(S_dixon_vals)))
for S_d, col in zip(S_dixon_vals, cols_d):
    inv_v_d = [1/v_noncompetitive(S_d, Vmax, Km, I, Ki_std) for I in I_dixon]
    ax5.plot(I_dixon, inv_v_d, color=col, label=f'[S]={S_d} mM')

ax5.axvline(-Ki_std, color='#E24B4A', linestyle='--', alpha=0.7,
             label=f'-Ki = -{Ki_std} uM')
ax5.set_xlabel('[I]  (uM)'); ax5.set_ylabel('1/v0  (min/umol)')
ax5.set_title('MSc Panel E: Dixon plot (1/v vs [I])\nAll lines meet at -Ki on X-axis', fontsize=10)
ax5.legend(fontsize=8)
ax5.text(0.6, 0.15,
          'Lines meet at x = -Ki\nregardless of [S]\n(non-competitive only)',
          transform=ax5.transAxes, fontsize=8.5, color='#534AB7',
          bbox=dict(boxstyle='round', facecolor='#EEEDFE', edgecolor='#7F77DD'))

# ── Panel F: MSc — distinguish comp vs non-comp from dose-response shape ─
ax6 = axes[1,2]
I_dr = np.logspace(-1, 3, 300)
S_test = Km   # test at [S] = Km
act_comp  = [v_competitive(S_test, Vmax, Km, I, Ki_std) /
              v_noinhib(S_test, Vmax, Km)*100 for I in I_dr]
act_nonc  = [v_noncompetitive(S_test, Vmax, Km, I, Ki_std) /
              v_noinhib(S_test, Vmax, Km)*100 for I in I_dr]
act_comp2 = [v_competitive(S_test*4, Vmax, Km, I, Ki_std) /
              v_noinhib(S_test*4, Vmax, Km)*100 for I in I_dr]
act_nonc2 = [v_noncompetitive(S_test*4, Vmax, Km, I, Ki_std) /
              v_noinhib(S_test*4, Vmax, Km)*100 for I in I_dr]
ax6.semilogx(I_dr, act_comp,  color='#7F77DD', lw=2,  label=f'Competitive [S]=Km={Km}')
ax6.semilogx(I_dr, act_comp2, color='#7F77DD', lw=2,  linestyle='--',
              label=f'Competitive [S]={Km*4} mM')
ax6.semilogx(I_dr, act_nonc,  color='#E24B4A', lw=2,  label=f'Non-comp [S]=Km={Km}')
ax6.semilogx(I_dr, act_nonc2, color='#E24B4A', lw=2,  linestyle='--',
              label=f'Non-comp [S]={Km*4} mM')
ax6.axhline(50, color='#aaa', linestyle=':', alpha=0.7)
ax6.set_xlabel('[I]  (uM, log)'); ax6.set_ylabel('% activity')
ax6.set_title('MSc Panel F: IC50 shift with [S] distinguishes types\nComp: IC50 shifts | Non-comp: IC50 stays', fontsize=9)
ax6.legend(fontsize=7.5)

plt.tight_layout()
plt.savefig('inh_s4_noncompetitive_deep.png', dpi=150, bbox_inches='tight')
plt.show()

print('NON-COMPETITIVE — PARAMETER SUMMARY:')
print(f'  True Ki = {Ki_std} uM')
print(f'  Ki from LB y-intercept plot = {Ki_nc:.1f} uM')
print()
print('BSc CHECK (answer in notebook):')
print('  1. Do all LB lines share the same x-intercept? Read it from Panel B.')
print('     x-intercept = ____  therefore Km = ____')
print('  2. At [I]=40 uM, Vmax_app = ____.')
print('     What fraction of enzyme molecules are inhibited at this concentration?')
print()
print('MSc EXTENSION:')
print('  1. LB y-intercept = 1/Vmax_app = (1/Vmax)*(1 + [I]/Ki)')
print(f'     Slope of y-int vs [I] = 1/(Vmax*Ki) = {1/(Vmax*Ki_std):.6f}')
print(f'     Measured from fit:       {coeffs_y[0]:.6f}   <- should match')
print()
print('  2. The Dixon plot (Panel E): ALL lines meet at x = -Ki')
print('     This is true for non-competitive inhibition only.')
print('     For competitive inhibition, Dixon plot lines meet ABOVE the x-axis.')
print('     This is a powerful diagnostic for mixed vs pure non-competitive.')


---
## Section 5 — Uncompetitive Inhibition
### The inhibitor waits for substrate to arrive

This is the most counterintuitive inhibition type — and the one most commonly  
misunderstood or skipped in undergraduate teaching.

The molecular story: the inhibitor **cannot bind free enzyme (E)**.  
It can only bind the **ES complex** — the conformation that forms after substrate  
has already entered the active site. The inhibitor's binding site is only  
exposed or created when the enzyme-substrate complex forms.

**The paradox:** *Adding more substrate makes inhibition WORSE.*  
More substrate → more ES complex → more inhibitor binding sites exposed → more inhibition.  
This is the opposite of competitive inhibition.

**Mathematical consequence:**
```
alpha' = 1 + [I]/Ki'
Vmax_app = Vmax / alpha'      (falls)
Km_app   = Km   / alpha'      (also falls — same factor)
```
Both Km and Vmax fall by exactly the same factor.  
The ratio Vmax/Km (catalytic efficiency) is unchanged — only the absolute values shift.

**Real examples:**  
Some herbicides on plant enzymes.  
Lithium on inositol monophosphatase.  
Certain pesticides on insect acetylcholinesterase.


In [ ]:
# === SECTION 5: Uncompetitive Inhibition ===

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Section 5 — Uncompetitive Inhibition: The Paradox of Substrate Worsening Inhibition',
             fontsize=13, fontweight='bold')

# Panel A: v vs [S]
ax1 = axes[0]
for I, col in zip(I_vals, cols):
    v = v_uncompetitive(S_range, Vmax, Km, I, Ki_std)
    alpha_p = 1+I/Ki_std
    lbl = f'[I]={I} uM  (Km_app={Km/alpha_p:.1f}, Vmax_app={Vmax/alpha_p:.0f})'
    ax1.plot(S_range, v, color=col, label=lbl)

ax1.set_xlabel('[S]  (mM)'); ax1.set_ylabel('v0  (umol/min)')
ax1.set_title('Both Vmax AND Km fall\nCurves shift down AND left', fontsize=11)
ax1.legend(fontsize=7)

# Panel B: Lineweaver-Burk — parallel lines
ax2 = axes[1]
for I, col in zip(I_vals, cols):
    v_lb = v_uncompetitive(S_lb, Vmax, Km, I, Ki_std)
    ax2.plot(1/S_lb, 1/v_lb, 'o-', color=col, markersize=4, label=f'[I]={I} uM')

ax2.axvline(0, color='#aaa', lw=0.8); ax2.axhline(0, color='#aaa', lw=0.8)
ax2.set_xlabel('1/[S]  (mM-1)'); ax2.set_ylabel('1/v0  (min/umol)')
ax2.set_title('Lineweaver-Burk: PARALLEL lines\n(same slope = same Vmax/Km ratio)', fontsize=11)
ax2.legend(fontsize=8)
ax2.text(0.55, 0.85, 'Parallel lines =\nuncompetitive!',
          transform=ax2.transAxes, fontsize=9, color='#BA7517',
          bbox=dict(boxstyle='round', facecolor='#FAEEDA', edgecolor='#BA7517'))

# Panel C: The three types compared on Lineweaver-Burk
ax3 = axes[2]
I_demo = 40
types = [
    ('No inhibitor',    lambda S: v_noinhib(S, Vmax, Km),              '#1D9E75', '-'),
    ('Competitive',     lambda S: v_competitive(S, Vmax, Km, I_demo, Ki_std),    '#7F77DD', '--'),
    ('Non-competitive', lambda S: v_noncompetitive(S, Vmax, Km, I_demo, Ki_std), '#D85A30', ':'),
    ('Uncompetitive',   lambda S: v_uncompetitive(S, Vmax, Km, I_demo, Ki_std),  '#BA7517', '-.'),
]
for lbl, fn, col, ls in types:
    v_lb = fn(S_lb)
    ax3.plot(1/S_lb, 1/v_lb, 'o'+ls, color=col, markersize=4, label=lbl, linewidth=1.8)

ax3.axvline(0, color='#aaa', lw=0.8); ax3.axhline(0, color='#aaa', lw=0.8)
ax3.set_xlabel('1/[S]  (mM-1)'); ax3.set_ylabel('1/v0  (min/umol)')
ax3.set_title('All three types on one Lineweaver-Burk\nThe intersection point is the diagnostic', fontsize=11)
ax3.legend(fontsize=8)

plt.tight_layout()
plt.savefig('inh_s5_uncompetitive.png', dpi=150, bbox_inches='tight')
plt.show()

print('UNCOMPETITIVE INHIBITION SUMMARY:')
for I in I_vals:
    a = 1+I/Ki_std
    print(f'  [I]={I:3d} uM  -> Vmax_app={Vmax/a:5.1f}  Km_app={Km/a:5.1f}  ratio={Vmax/Km:.2f} (unchanged)')
print()
print('DIAGNOSTIC SIGNATURE: Parallel lines on Lineweaver-Burk')
print('THE PARADOX:          More substrate = more ES = more I binding = more inhibition')
print('PRACTICAL NOTE:       Rarely seen in isolation -- usually part of mixed inhibition')


---
## Section 6 — Mixed Inhibition
### The general case that contains all others

Mixed inhibition is the most general — and the most biologically realistic — case.  
The inhibitor binds both free enzyme (E) and the ES complex, but with **different affinities**.

- Ki = inhibitor's affinity for free E
- Ki' = inhibitor's affinity for ES complex

When Ki' >> Ki (inhibitor strongly prefers free E) → approaches competitive inhibition  
When Ki >> Ki' (inhibitor strongly prefers ES) → approaches uncompetitive inhibition  
When Ki = Ki' (equal affinity for E and ES) → pure non-competitive inhibition

**Mathematical consequence:**
```
alpha   = 1 + [I]/Ki       (effect on free E)
alpha'  = 1 + [I]/Ki'      (effect on ES)
v = Vmax * [S] / (alpha*Km + alpha'*[S])
```
Vmax always falls. Km may rise or fall depending on whether alpha > alpha' or vice versa.

**Why this matters for Western Odisha soil enzymes:**  
Heavy metal inhibition of soil enzymes is rarely pure competitive or pure non-competitive.  
It is almost always mixed — the metal ion can coordinate to multiple sites on the enzyme  
protein. Correctly identifying mixed vs non-competitive requires experiments at multiple  
[I] values, not just a single comparison.


In [ ]:
# === SECTION 6: Mixed Inhibition — the general case ===

fig, axes = plt.subplots(2, 2, figsize=(13, 10))
fig.suptitle('Section 6 — Mixed Inhibition: The General Case',
             fontsize=13, fontweight='bold')

I_demo = 30

# Panel A: vary Ki'/Ki ratio — show how mixed approaches other types
ax1 = axes[0,0]
scenarios_mix = [
    dict(label='Ki=Ki-prime=20 (non-competitive)', Ki=20, Ki_p=20,  color='#D85A30'),
    dict(label='Ki=20, Ki-prime=5 (uncompetitive-like)', Ki=20, Ki_p=5,   color='#BA7517'),
    dict(label='Ki=5,  Ki-prime=20 (competitive-like)',  Ki=5,  Ki_p=20,  color='#7F77DD'),
    dict(label='Ki=10, Ki-prime=40 (mixed, prefers E)',  Ki=10, Ki_p=40,  color='#E24B4A'),
]
v_ref = v_noinhib(S_range, Vmax, Km)
ax1.plot(S_range, v_ref, color='#1D9E75', linewidth=2.5, label='No inhibitor')
for sc in scenarios_mix:
    v = v_mixed(S_range, Vmax, Km, I_demo, sc['Ki'], sc['Ki_p'])
    ax1.plot(S_range, v, color=sc['color'], linewidth=1.8, linestyle='--', label=sc['label'])
ax1.set_xlabel('[S]  (mM)'); ax1.set_ylabel('v0  (umol/min)')
ax1.set_title(f'Mixed inhibition: varying Ki/Ki-prime ratio\n[I]={I_demo} uM in all cases', fontsize=10)
ax1.legend(fontsize=7)

# Panel B: Lineweaver-Burk for mixed
ax2 = axes[0,1]
I_mix_vals = [0, 15, 30, 60]
cols_mix = ['#1D9E75','#9FE1CB','#D85A30','#E24B4A']
Ki_mix, Ki_prime_mix = 15, 45
for I, col in zip(I_mix_vals, cols_mix):
    v_lb = v_mixed(S_lb, Vmax, Km, I, Ki_mix, Ki_prime_mix)
    ax2.plot(1/S_lb, 1/v_lb, 'o-', color=col, markersize=4, label=f'[I]={I} uM')
ax2.axvline(0, color='#aaa', lw=0.8); ax2.axhline(0, color='#aaa', lw=0.8)
ax2.set_xlabel('1/[S]  (mM-1)'); ax2.set_ylabel('1/v0  (min/umol)')
ax2.set_title('Mixed inhibition Lineweaver-Burk:\nLines intersect LEFT of Y-axis', fontsize=11)
ax2.legend(fontsize=8)
ax2.text(0.55, 0.15, 'Intersection in\n2nd quadrant',
          transform=ax2.transAxes, fontsize=9, color='#D85A30',
          bbox=dict(boxstyle='round', facecolor='#FAECE7', edgecolor='#D85A30'))

# Panel C: The complete diagnostic table — all four types
ax3 = axes[1,0]
ax3.axis('off')
table_data = [
    ['Type', 'Vmax', 'Km', 'LB intersection', 'Substrate test'],
    ['Competitive',    'Unchanged', 'Rises',     'Y-axis',         'Rescues rate'],
    ['Non-competitive','Falls',     'Unchanged', 'X-axis',         'No rescue'],
    ['Uncompetitive',  'Falls',     'Falls',     'Parallel lines', 'Worsens'],
    ['Mixed',          'Falls',     'Changes',   '2nd quadrant',   'Partial rescue'],
]
row_colors = ['#E1F5EE','#EEEDFE','#FCEBEB','#FAEEDA','#FAECE7']
table = ax3.table(cellText=table_data[1:], colLabels=table_data[0],
                   cellLoc='center', loc='center',
                   cellColours=[[row_colors[i+1]]*5 for i in range(4)])
table.auto_set_font_size(False); table.set_fontsize(9)
table.scale(1, 2.2)
ax3.set_title('Complete Diagnostic Table\n(the key to reading inhibition from data)', fontsize=11)

# Panel D: Complete Lineweaver-Burk comparison — all four types
ax4 = axes[1,1]
I_d = 40
all_types = [
    ('No inhibitor',    lambda S: v_noinhib(S, Vmax, Km),                        '#1D9E75', '-',  3.0),
    ('Competitive',     lambda S: v_competitive(S, Vmax, Km, I_d, Ki_std),       '#7F77DD', '--', 2.0),
    ('Non-competitive', lambda S: v_noncompetitive(S, Vmax, Km, I_d, Ki_std),    '#D85A30', ':',  2.0),
    ('Uncompetitive',   lambda S: v_uncompetitive(S, Vmax, Km, I_d, Ki_std),     '#BA7517', '-.', 2.0),
    ('Mixed',           lambda S: v_mixed(S, Vmax, Km, I_d, 15, 45),             '#E24B4A', '--', 2.0),
]
for lbl, fn, col, ls, lw in all_types:
    v_lb = fn(S_lb)
    ax4.plot(1/S_lb, 1/v_lb, 'o'+ls, color=col, markersize=4,
              label=lbl, linewidth=lw)
ax4.axvline(0, color='#aaa', lw=0.8); ax4.axhline(0, color='#aaa', lw=0.8)
ax4.set_xlabel('1/[S]  (mM-1)'); ax4.set_ylabel('1/v0  (min/umol)')
ax4.set_title('All four types: Lineweaver-Burk master diagram\nRead the intersection to identify type', fontsize=10)
ax4.legend(fontsize=8)

plt.tight_layout()
plt.savefig('inh_s6_mixed.png', dpi=150, bbox_inches='tight')
plt.show()

print('THE MASTER DIAGNOSTIC RULE:')
print('  Look at where the Lineweaver-Burk lines intersect:')
print('  Y-axis        -> Competitive    (Vmax shared, Km differs)')
print('  X-axis        -> Non-competitive (Km shared, Vmax differs)')
print('  Parallel      -> Uncompetitive  (same slope = same Vmax/Km)')
print('  2nd quadrant  -> Mixed          (neither x nor y axis)')


---
## Section 7 — The Lineweaver-Burk as a Detective Tool
### From unknown data to inhibition type and Ki value

> **BSc:** Identify the inhibition type from the LB intersection pattern.
> Use the diagnostic key from Section 6. Write your answer before running the reveal.

> **MSc:** Additionally extract Ki from each dataset using the quantitative methods
> from Sections 3 and 4 (slope vs [I] for competitive, y-intercept vs [I] for non-competitive).

---
### The four diagnostic signatures — your reference

```
Competitive:     Lines meet at Y-axis     -> Km rises,  Vmax unchanged
Non-competitive: Lines meet at X-axis     -> Km unchanged, Vmax falls
Uncompetitive:   Lines are PARALLEL       -> Both fall equally
Mixed:           Lines meet in 2nd quadrant -> Vmax falls, Km may change
```

**Practical note on real data:**  
Real LB lines rarely intersect perfectly. Experimental noise shifts each point slightly.  
You must fit a line to each [I] series first, then find the mathematical intersection  
of the fitted lines — not eyeball where scattered points seem to meet.  
The cell below includes realistic noise and shows how to handle it properly.


In [ ]:
# === SECTION 7: LB Diagnostic — Unknown Data with Realistic Noise ===
# BSc:  identify type from LB pattern
# MSc:  extract Ki quantitatively from the fitted lines

np.random.seed(42)
S_exp = np.array([5, 8, 12, 20, 35, 60, 100, 160])
noise = 2.2

unknown_datasets = {
    'Unknown A': dict(
        data={I: np.clip(v_competitive(S_exp,Vmax,Km,I,18)+np.random.normal(0,noise,len(S_exp)),0.5,None)
              for I in [0, 25, 50]},
        true_type='Competitive', true_Ki=18, color='#7F77DD'),
    'Unknown B': dict(
        data={I: np.clip(v_noncompetitive(S_exp,Vmax,Km,I,22)+np.random.normal(0,noise,len(S_exp)),0.5,None)
              for I in [0, 25, 50]},
        true_type='Non-competitive', true_Ki=22, color='#D85A30'),
    'Unknown C': dict(
        data={I: np.clip(v_uncompetitive(S_exp,Vmax,Km,I,25)+np.random.normal(0,noise,len(S_exp)),0.5,None)
              for I in [0, 25, 50]},
        true_type='Uncompetitive', true_Ki=25, color='#BA7517'),
    'Unknown D': dict(
        data={I: np.clip(v_mixed(S_exp,Vmax,Km,I,12,35)+np.random.normal(0,noise,len(S_exp)),0.5,None)
              for I in [0, 25, 50]},
        true_type='Mixed', true_Ki=12, color='#E24B4A'),
}
I_series = [0, 25, 50]
inh_line_cols = ['#1D9E75','#888780','#444441']

# ── Figure 1: LB plots (BSc identification task) ───────────────
fig, axes = plt.subplots(2, 2, figsize=(13, 10))
fig.suptitle('Section 7 — BSc Task: Identify the Inhibition Type\n'
             '(Write your answers before running the reveal cell)',
             fontsize=12, fontweight='bold')

fitted_lines = {}   # store for MSc analysis
for ax, (name, ds) in zip(axes.flatten(), unknown_datasets.items()):
    lines_this = {}
    for (I, v_data), lc in zip(ds['data'].items(), inh_line_cols):
        inv_s = 1/S_exp
        inv_v = 1/v_data
        ax.scatter(inv_s, inv_v, color=ds['color'] if I==0 else lc,
                    s=40, zorder=5)
        coeffs = np.polyfit(inv_s, inv_v, 1)
        lines_this[I] = coeffs
        x_line = np.linspace(-0.06, max(inv_s)*1.05, 200)
        ax.plot(x_line, np.polyval(coeffs, x_line),
                 color=ds['color'] if I==0 else lc,
                 linewidth=1.8, label=f'[I]={I} uM')
    fitted_lines[name] = lines_this
    ax.axvline(0, color='#aaa', lw=0.8); ax.axhline(0, color='#aaa', lw=0.8)
    ax.set_xlim(-0.07, 0.22)
    ax.set_xlabel('1/[S]  (mM⁻¹)'); ax.set_ylabel('1/v0  (min/umol)')
    ax.set_title(f'{name}', fontsize=11, color=ds['color'], fontweight='bold')
    ax.legend(fontsize=8)
    ax.text(0.04, 0.92, 'Your answer: ___________\nKi ~ ___ uM',
             transform=ax.transAxes, fontsize=8.5, color='#333',
             bbox=dict(boxstyle='round', facecolor='lightyellow', edgecolor='#888'))

plt.tight_layout()
plt.savefig('inh_s7_bsc_task.png', dpi=150, bbox_inches='tight')
plt.show()

print('BSc TASK: Write your identification before running Section 7b')
print('  Unknown A: type = _______   Ki ~ _______ uM')
print('  Unknown B: type = _______   Ki ~ _______ uM')
print('  Unknown C: type = _______   Ki ~ _______ uM')
print('  Unknown D: type = _______   Ki ~ _______ uM')
print()
print('Hint: look at the fitted lines (not the scattered points).')
print('Find where the fitted lines would intersect if extended.')
print('That intersection location is your diagnostic.')


In [ ]:
# === SECTION 7b: Reveal + MSc Ki Extraction ===
# Run ONLY after writing BSc answers above

print('ANSWERS AND MSc Ki EXTRACTION:')
print('='*55)

fig, axes = plt.subplots(2, 2, figsize=(13, 10))
fig.suptitle('Section 7b — MSc: Quantitative Ki Extraction from Each Dataset',
             fontsize=12, fontweight='bold')

for ax, (name, ds) in zip(axes.flatten(), unknown_datasets.items()):
    print(f'\n{name}: TRUE TYPE = {ds["true_type"]}  |  True Ki = {ds["true_Ki"]} uM')

    I_list = list(ds['data'].keys())
    coeffs_list = [fitted_lines[name][I] for I in I_list]
    slopes = [c[0] for c in coeffs_list]   # slope = Km_app/Vmax
    y_ints = [c[1] for c in coeffs_list]   # y-int = 1/Vmax_app

    # Find intersection of [I]=0 and [I]=50 lines
    c0, c1 = coeffs_list[0], coeffs_list[-1]
    if abs(c0[0]-c1[0]) > 1e-6:
        x_int = (c1[1]-c0[1])/(c0[0]-c1[0])
        y_int_pt = np.polyval(c0, x_int)
    else:
        x_int, y_int_pt = None, None

    # Determine diagnostic region
    if x_int is not None:
        if   abs(x_int) < 0.008:          region = 'Y-axis  -> Competitive'
        elif abs(y_int_pt) < 0.004:       region = 'X-axis  -> Non-competitive'
        elif x_int < -0.005:              region = '2nd quadrant -> Mixed'
        else:                             region = 'uncertain'
    else:
        region = 'Parallel -> Uncompetitive'

    print(f'  LB intersection region: {region}')

    # MSc: extract Ki from slopes (competitive) or y-intercepts (non-competitive)
    if ds['true_type'] == 'Competitive':
        slope_coeffs = np.polyfit(I_list, slopes, 1)
        Ki_calc = -slope_coeffs[1]/slope_coeffs[0]
        print(f'  Ki from slope vs [I]: {Ki_calc:.1f} uM  (true = {ds["true_Ki"]})')
    elif ds['true_type'] == 'Non-competitive':
        yint_coeffs = np.polyfit(I_list, y_ints, 1)
        Ki_calc = -yint_coeffs[1]/yint_coeffs[0]
        print(f'  Ki from y-int vs [I]: {Ki_calc:.1f} uM  (true = {ds["true_Ki"]})')
    elif ds['true_type'] == 'Uncompetitive':
        yint_coeffs = np.polyfit(I_list, y_ints, 1)
        Ki_p_calc = -yint_coeffs[1]/yint_coeffs[0]
        print(f'  Ki-prime from y-int vs [I]: {Ki_p_calc:.1f} uM  (true = {ds["true_Ki"]})')
    else:
        print(f'  Mixed: requires fitting both alpha and alpha-prime separately')

    # Plot on figure
    inh_lc = ['#1D9E75','#888780','#444441']
    for (I, v_data), lc in zip(ds['data'].items(), inh_lc):
        inv_s = 1/S_exp; inv_v = 1/v_data
        ax.scatter(inv_s, inv_v, color=lc, s=35, zorder=5)
        c = fitted_lines[name][I]
        x_line = np.linspace(-0.07, max(inv_s)*1.05, 200)
        ax.plot(x_line, np.polyval(c, x_line), color=ds['color'] if I==0 else lc,
                 linewidth=1.8, label=f'[I]={I}')
    if x_int is not None and y_int_pt is not None:
        ax.scatter([x_int],[y_int_pt], color='#E24B4A', s=120, zorder=6,
                    marker='*', label='Intersection')
    ax.axvline(0, color='#aaa', lw=0.8); ax.axhline(0, color='#aaa', lw=0.8)
    ax.set_xlim(-0.07, 0.22)
    ax.set_xlabel('1/[S]  (mM⁻¹)'); ax.set_ylabel('1/v0  (min/umol)')
    ax.set_title(f'{name}: {ds["true_type"]}', fontsize=11,
                  color=ds['color'], fontweight='bold')
    ax.legend(fontsize=7.5)
    ax.text(0.04, 0.92, f'True type: {ds["true_type"]}\nTrue Ki = {ds["true_Ki"]} uM',
             transform=ax.transAxes, fontsize=8.5, color=ds['color'],
             bbox=dict(boxstyle='round', facecolor='white', edgecolor=ds['color']))

plt.tight_layout()
plt.savefig('inh_s7b_msc_reveal.png', dpi=150, bbox_inches='tight')
plt.show()
print()
print('MSc REFLECTION:')
print('  How close was your Ki estimate to the true value?')
print('  The error comes from: measurement noise, limited number of [S] points,')
print('  and the inherent amplification of error in 1/v vs 1/[S] transformations.')
print('  This is why modern analysis uses nonlinear regression on v vs [S] directly')
print('  (scipy.optimize.curve_fit) rather than Lineweaver-Burk for final parameter estimates.')
print('  The LB plot remains essential for TYPE identification. Ki is better from nonlinear fit.')


---
## Section 8 — Reversible vs Irreversible Inhibition
### Heavy metals as a special case — permanent damage vs temporary competition

Everything in Sections 3–7 assumed the inhibitor binds **reversibly**.  
Remove the inhibitor — dilute the sample, dialyse, add a chelating agent — and activity returns.

**Irreversible inhibitors are different. They form a covalent bond with the enzyme.**  
They do not dissociate. The enzyme molecule is permanently inactivated.

This distinction matters enormously for your soil biomarker work:

| Property | Reversible | Irreversible |
|----------|-----------|-------------|
| Bond type | Non-covalent (H-bond, ionic, hydrophobic) | Covalent |
| Kinetic signature | Normal Km and Vmax changes with [I] | Apparent Vmax falls with time even at fixed [I] |
| Recovery test | Dialysis or dilution restores activity | No recovery after removal of inhibitor |
| Heavy metal example | Low-level Zn2+ (some reversible coordination) | Hg2+, Pb2+ at high concentrations |
| Ecological meaning | Temporary stress — enzyme recovers if contamination removed | Permanent damage — enzyme pool depleted |
| Experimental test | Activity vs [I] at equilibrium | Activity vs preincubation time at fixed [I] |

**The time-dependence test for irreversible inhibition:**  
Incubate enzyme with inhibitor for 0, 5, 10, 20, 30 minutes **before** adding substrate.  
If inhibition is irreversible, rate falls progressively with preincubation time even  
though [I] is the same in all tubes. If reversible, preincubation time makes no difference.


In [ ]:
# === SECTION 8: Reversible vs Irreversible Inhibition ===

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Section 8 — Reversible vs Irreversible: The Critical Distinction for Soil Biomarkers',
             fontsize=12, fontweight='bold')

# Panel A: reversible — activity recovers after dilution
ax1 = axes[0]
I_revs = [0, 20, 40, 60, 80]
for I, col in zip(I_revs, cols):
    v = v_noncompetitive(S_range, Vmax, Km, I, Ki_std)
    ax1.plot(S_range, v, color=col, label=f'[I]={I} uM')
ax1.set_xlabel('[S]  (mM)'); ax1.set_ylabel('v0  (umol/min)')
ax1.set_title('Reversible inhibition\nEach curve = equilibrium at that [I]', fontsize=10)
ax1.legend(fontsize=8)
ax1.text(0.05, 0.12, 'Remove inhibitor\n-> activity returns',
          transform=ax1.transAxes, fontsize=9, color='#085041',
          bbox=dict(boxstyle='round', facecolor='#E1F5EE', edgecolor='#1D9E75'))

# Panel B: irreversible — activity falls with preincubation time
ax2 = axes[1]
preincub_times = [0, 5, 10, 20, 30, 60]   # minutes
k_irrev = 0.03   # inactivation rate constant (min-1)
cols_time = plt.cm.Reds(np.linspace(0.3, 0.9, len(preincub_times)))
for t_pre, col in zip(preincub_times, cols_time):
    Vmax_remaining = Vmax * np.exp(-k_irrev * t_pre)
    v = v_noinhib(S_range, Vmax_remaining, Km)
    ax2.plot(S_range, v, color=col,
              label=f't_pre={t_pre} min  (Vmax={Vmax_remaining:.0f})')
ax2.set_xlabel('[S]  (mM)'); ax2.set_ylabel('v0  (umol/min)')
ax2.set_title('Irreversible inhibition\nVmax falls with preincubation time', fontsize=10)
ax2.legend(fontsize=7.5)
ax2.text(0.05, 0.12, 'Remove inhibitor\n-> NO recovery',
          transform=ax2.transAxes, fontsize=9, color='#791F1F',
          bbox=dict(boxstyle='round', facecolor='#FCEBEB', edgecolor='#E24B4A'))

# Panel C: the time-dependence test — key diagnostic
ax3 = axes[2]
t_arr = np.linspace(0, 60, 200)
S_fixed = 100  # mM

# Reversible: rate is same regardless of preincubation time
I_rev_demo = 40
v_rev_const = v_noncompetitive(S_fixed, Vmax, Km, I_rev_demo, Ki_std)
ax3.plot(t_arr, np.full_like(t_arr, v_rev_const), color='#1D9E75',
          linewidth=2.5, label=f'Reversible [I]={I_rev_demo} uM')

# Irreversible: rate falls with preincubation time
for k, col, lbl in [
    (0.02, '#D85A30', 'Irreversible k=0.02'),
    (0.04, '#E24B4A', 'Irreversible k=0.04'),
]:
    Vmax_t = Vmax * np.exp(-k * t_arr)
    v_t = v_noinhib(S_fixed, Vmax_t, Km)
    ax3.plot(t_arr, v_t, color=col, linewidth=2.2, linestyle='--', label=lbl)

ax3.set_xlabel('Preincubation time with inhibitor (min)')
ax3.set_ylabel(f'v0 at [S]={S_fixed} mM  (umol/min)')
ax3.set_title('The preincubation test:\nFlat line = reversible | Falling line = irreversible', fontsize=10)
ax3.legend(fontsize=8.5)

plt.tight_layout()
plt.savefig('inh_s8_reversible.png', dpi=150, bbox_inches='tight')
plt.show()

print('CRITICAL DISTINCTION FOR OSHEC SOIL WORK:')
print('  Reversible inhibition by heavy metals:')
print('    -> If contamination is remediated, enzyme activity can recover')
print('    -> Km and Vmax return toward normal as metal is chelated/removed')
print()
print('  Irreversible inhibition by heavy metals (Hg2+, high Pb2+):')
print('    -> Enzyme molecules are permanently damaged')
print('    -> Activity only recovers when NEW enzyme is synthesised by soil microbes')
print('    -> This is a measure of chronic vs acute contamination impact')
print()
print('  EXPERIMENTAL DESIGN:')
print('  To test reversibility in a soil enzyme assay:')
print('  1. Incubate soil extract + metal ion for 0, 10, 30, 60 min')
print('  2. Add EDTA to chelate free metal ions (removes reversible component)')
print('  3. Measure remaining activity')
print('  4. If activity still reduced after EDTA -> irreversible component present')


---
## Section 9 — IC₅₀ and the Dose-Response Curve
### How potent is the inhibitor? Connecting kinetics to pharmacology and ecotoxicology

So far we have described *type* of inhibition.  
Now we ask: *how much inhibitor is needed to have a significant effect?*

**IC₅₀ (half-maximal inhibitory concentration):** the concentration of inhibitor  
that reduces enzyme activity to 50% of its uninhibited value, at a defined [S].

**Why IC₅₀ matters:**
- Pharmacology: drugs are ranked by IC₅₀ against their target enzyme
- Ecotoxicology: soil enzyme IC₅₀ values for specific heavy metals tell you
  how contaminated a soil must be before enzyme function is halved
- Biomarker sensitivity: a lower IC₅₀ means the enzyme responds to contamination
  at lower metal concentrations — more sensitive as a biomarker

**Important caution:** IC₅₀ depends on [S].  
For competitive inhibitors, raising [S] raises IC₅₀ (substrate partially protects the enzyme).  
For non-competitive inhibitors, IC₅₀ is independent of [S].  
Always report the [S] at which IC₅₀ was measured.

**The relationship between IC₅₀ and Ki (Cheng-Prusoff equation):**
```
Competitive:     IC50 = Ki * (1 + [S]/Km)
Non-competitive: IC50 = Ki
```


In [ ]:
# === SECTION 9: IC50 and Dose-Response ===

I_range_dr = np.logspace(-1, 3, 300)  # 0.1 to 1000 uM
S_fixed_vals = [5, 25, 50, 100, 200]  # different [S] values
cols_s = plt.cm.Blues(np.linspace(0.35, 0.95, len(S_fixed_vals)))

fig, axes = plt.subplots(2, 2, figsize=(13, 10))
fig.suptitle('Section 9 — IC50 and Dose-Response Curves\n'
             'Connecting Inhibition Constants to Practical Potency',
             fontsize=13, fontweight='bold')

# Panel A: dose-response for competitive inhibitor at different [S]
ax1 = axes[0,0]
for S_fix, col in zip(S_fixed_vals, cols_s):
    activities = [v_competitive(S_fix, Vmax, Km, I, Ki_std) /
                   v_noinhib(S_fix, Vmax, Km) * 100
                  for I in I_range_dr]
    ax1.semilogx(I_range_dr, activities, color=col,
                  label=f'[S]={S_fix} mM')

ax1.axhline(50, color='#E24B4A', linestyle='--', alpha=0.7, label='50% activity (IC50)')
ax1.set_xlabel('[I]  (uM, log scale)'); ax1.set_ylabel('% activity remaining')
ax1.set_title('Competitive: IC50 rises with [S]\n([S] protects enzyme from competitive inhibitor)', fontsize=10)
ax1.legend(fontsize=8)

# Panel B: dose-response for non-competitive — [S]-independent
ax2 = axes[0,1]
for S_fix, col in zip(S_fixed_vals, cols_s):
    activities = [v_noncompetitive(S_fix, Vmax, Km, I, Ki_std) /
                   v_noinhib(S_fix, Vmax, Km) * 100
                  for I in I_range_dr]
    ax2.semilogx(I_range_dr, activities, color=col,
                  label=f'[S]={S_fix} mM')
ax2.axhline(50, color='#E24B4A', linestyle='--', alpha=0.7, label='50% = IC50')
ax2.set_xlabel('[I]  (uM, log scale)'); ax2.set_ylabel('% activity remaining')
ax2.set_title('Non-competitive: IC50 = Ki (independent of [S])\nAll curves identical', fontsize=10)
ax2.legend(fontsize=8)

# Panel C: IC50 vs [S] — Cheng-Prusoff relationship
ax3 = axes[1,0]
S_arr_cp = np.linspace(0, 200, 300)
IC50_comp  = Ki_std * (1 + S_arr_cp / Km)
IC50_nonc  = np.full_like(S_arr_cp, Ki_std)
ax3.plot(S_arr_cp, IC50_comp, color='#7F77DD', linewidth=2.5,
          label=f'Competitive: IC50 = Ki*(1+[S]/Km)')
ax3.plot(S_arr_cp, IC50_nonc, color='#D85A30', linewidth=2.5, linestyle='--',
          label=f'Non-competitive: IC50 = Ki = {Ki_std} uM')
ax3.axvline(Km, color='#1D9E75', linestyle=':', alpha=0.6, label=f'Km = {Km} mM')
ax3.set_xlabel('[S]  (mM)'); ax3.set_ylabel('IC50  (uM)')
ax3.set_title('Cheng-Prusoff: how IC50 depends on [S]\n(always report [S] when citing IC50)', fontsize=10)
ax3.legend(fontsize=9)

# Panel D: Comparing IC50 for different heavy metals on soil urease
ax4 = axes[1,1]
# Literature-based IC50 values for heavy metals on soil urease
metals  = ['Cd2+', 'Pb2+', 'Hg2+', 'Cu2+', 'Zn2+', 'Cr6+']
IC50_vals = [85, 120, 8, 35, 210, 65]   # uM — approximate from literature
bar_colors = ['#BA7517','#D85A30','#E24B4A','#7F77DD','#1D9E75','#888780']
bars = ax4.bar(metals, IC50_vals, color=bar_colors, alpha=0.85, edgecolor='white')
ax4.set_ylabel('IC50 for soil urease  (uM)')
ax4.set_title('Heavy metal potency on soil urease\nLower IC50 = more potent inhibitor = better biomarker', fontsize=10)
for bar, val in zip(bars, IC50_vals):
    ax4.text(bar.get_x()+bar.get_width()/2, bar.get_height()+2,
              str(val), ha='center', fontsize=9, fontweight='bold')
ax4.text(0.98, 0.92, 'Hg2+ most potent\n(lowest IC50)',
          transform=ax4.transAxes, ha='right', fontsize=9, color='#E24B4A',
          bbox=dict(boxstyle='round', facecolor='#FCEBEB', edgecolor='#E24B4A'))

plt.tight_layout()
plt.savefig('inh_s9_ic50.png', dpi=150, bbox_inches='tight')
plt.show()

print('IC50 SUMMARY FOR SOIL BIOMARKER APPLICATIONS:')
for m, ic in zip(metals, IC50_vals):
    print(f'  {m:5s}: IC50 ~ {ic:4d} uM  (soil urease, approx. from literature)')
print()
print('NOTE: IC50 values vary with soil type, pH, organic matter, and enzyme source.')
print('For your OSHEC sites: measure IC50 of site-specific soil enzymes against')
print('the metals actually present in Jharsuguda/Rourkela industrial zone soils.')
print('A shift in IC50 between sites tells you both contamination level AND enzyme sensitivity.')


---
## 🌍 Section 10 — Western Odisha: Heavy Metals as Soil Enzyme Inhibitors
### Reading contamination history from inhibition patterns

The industrial zones of Western Odisha — Jharsuguda, Rourkela, Sundargarh —  
release a complex mixture of heavy metals into surrounding soils:  
Pb2+, Cd2+, Hg2+, Cu2+, Zn2+, Cr6+, and Fe2+/Fe3+.

Each metal has a characteristic inhibition profile across different soil enzymes.  
The **pattern of enzyme inhibition** — which enzyme is most affected, what type of  
inhibition, reversible or not — is a fingerprint of the contamination source.

**The OSHEC research question in kinetic terms:**  
At your earthworm collection sites, does the soil enzyme inhibition pattern:
1. Show elevated apparent Km (suggesting competitive/mixed inhibition by metals mimicking substrate)?
2. Show reduced Vmax with unchanged Km (suggesting non-competitive binding to cofactor sites)?
3. Show time-dependent loss of activity (suggesting irreversible modification)?
4. Show IC₅₀ values that correlate with measured metal concentrations in soil?

Each answer points to a different molecular mechanism — and a different remediation strategy.


In [ ]:
# === SECTION 10: Western Odisha Contamination Profiles ===

# Hypothetical kinetic profiles across a contamination gradient
# Replace with real field + lab data from OSHEC project

sites = {
    'Forest control\n(Kuchinda)':         dict(Vmax=92,  Km=16,  inh_type='none',          Ki=None, color='#1D9E75'),
    'Agricultural\n(5 km buffer)':        dict(Vmax=78,  Km=19,  inh_type='competitive',    Ki=180,  color='#639922'),
    'Peri-industrial\n(2 km)':            dict(Vmax=61,  Km=38,  inh_type='mixed',          Ki=55,   color='#BA7517'),
    'Industrial edge\n(0.5 km)':          dict(Vmax=42,  Km=44,  inh_type='noncompetitive', Ki=30,   color='#D85A30'),
    'Contamination\nepicenter':           dict(Vmax=19,  Km=48,  inh_type='irreversible',   Ki=None, color='#E24B4A'),
}

fig, axes = plt.subplots(2, 2, figsize=(14, 11))
fig.suptitle('Section 10 — Western Odisha Soil Urease Inhibition Profiles\n'
             'Each site tells a different story about contamination type and severity',
             fontsize=12, fontweight='bold')

S_r = np.linspace(0.1, 200, 400)

# Panel A: MM curves
ax1 = axes[0,0]
for site, p in sites.items():
    v = v_noinhib(S_r, p['Vmax'], p['Km'])
    ax1.plot(S_r, v, color=p['color'], linewidth=2.2,
              label=site.replace(chr(10),' '))
ax1.set_xlabel('[Urea]  (mM)'); ax1.set_ylabel('Urease activity  (umol/min/g)')
ax1.set_title('Apparent MM curves along contamination gradient\n'
               '(fitted to observed v0 data from each site)', fontsize=10)
ax1.legend(fontsize=7.5)

# Panel B: Km and Vmax as site biomarkers
site_names = [s.split(chr(10))[0] for s in sites]
Kms_s  = [p['Km']   for p in sites.values()]
Vmaxs_s= [p['Vmax'] for p in sites.values()]
cols_s = [p['color'] for p in sites.values()]
x = np.arange(len(sites))

ax2 = axes[0,1]
b1 = ax2.bar(x-0.2, Kms_s,   0.38, color=cols_s, alpha=0.85, label='Km (mM)',  edgecolor='white')
b2 = ax2.bar(x+0.2, Vmaxs_s, 0.38, color=cols_s, alpha=0.45, label='Vmax', edgecolor='white', hatch='//')
ax2.set_xticks(x); ax2.set_xticklabels(site_names, fontsize=8, rotation=20, ha='right')
ax2.set_ylabel('Value'); ax2.legend(fontsize=9)
ax2.set_title('Km rises and Vmax falls\nalong contamination gradient', fontsize=10)
for bar, v in zip(b1, Kms_s):  ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1, str(v), ha='center', fontsize=7)
for bar, v in zip(b2, Vmaxs_s): ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1, str(v), ha='center', fontsize=7)

# Panel C: Catalytic efficiency as integrated index
ax3 = axes[1,0]
effs = [p['Vmax']/p['Km'] for p in sites.values()]
bars3 = ax3.bar(site_names, effs, color=cols_s, alpha=0.85, edgecolor='white')
ax3.set_ylabel('Vmax/Km  (catalytic efficiency)')
ax3.set_title('Catalytic efficiency as single\nintegrated soil health index', fontsize=10)
ax3.tick_params(axis='x', rotation=20)
for bar, v in zip(bars3, effs):
    ax3.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.05,
              f'{v:.1f}', ha='center', fontsize=9, fontweight='bold')

# Panel D: Inhibition type guide per site
ax4 = axes[1,1]
ax4.axis('off')
inh_table = [
    ['Site', 'Inhibition type', 'Metal implicated', 'Remediation signal'],
    ['Forest control',     'None',           'Background only',  'Baseline reference'],
    ['Agricultural',       'Competitive',    'Low Zn2+/Cu2+',   'Reversible, recoverable'],
    ['Peri-industrial',    'Mixed',          'Pb2+/Cd2+ mixture','Partial recovery possible'],
    ['Industrial edge',    'Non-competitive','Hg2+/high Pb2+',  'Slow recovery'],
    ['Epicenter',          'Irreversible',   'High Hg2+',        'Enzyme pool depleted'],
]
row_cols = ['#E1F5EE','#EAF3DE','#FAEEDA','#FAECE7','#FCEBEB','#FCEBEB']
tbl = ax4.table(cellText=inh_table[1:], colLabels=inh_table[0],
                 cellLoc='center', loc='center',
                 cellColours=[[row_cols[i+1]]*4 for i in range(5)])
tbl.auto_set_font_size(False); tbl.set_fontsize(8)
tbl.scale(1, 2.0)
ax4.set_title('Inhibition type guides remediation strategy\n(replace with real OSHEC field + lab data)', fontsize=10)

plt.tight_layout()
plt.savefig('inh_s10_western_odisha.png', dpi=150, bbox_inches='tight')
plt.show()

print('RESEARCH DESIGN FOR OSHEC SITES:')
print('  For each earthworm collection site, measure:')
print('  1. MM curve (8 [S] values, fresh tube each) -> Km, Vmax')
print('  2. Inhibition type (add metal at 3 concentrations, LB plot) -> competitive/non-comp/mixed')
print('  3. Reversibility (preincubation test + EDTA rescue) -> reversible/irreversible')
print('  4. IC50 for dominant site metals -> potency ranking')
print()
print('  This gives a complete kinetic fingerprint of soil enzyme health at each site.')
print('  Combined with heavy metal speciation data and earthworm genomic responses,')
print('  it forms the foundation of your biomarker panel.')


---
## Section 11 — 🧪 Your Own Inhibition Experiment
### Design, run, and diagnose

**Instructions:**  
1. Choose your inhibitor type and concentration  
2. Write your prediction for Km, Vmax, and LB pattern BEFORE running  
3. Run the cell  
4. Compare prediction with output  
5. Calculate Ki from the data


In [ ]:
# ============================================================
# STUDENT EXPERIMENT -- MODIFY AND RE-RUN
# ============================================================

# --- Your enzyme ---
MY_Vmax = 80.0
MY_Km   = 25.0

# --- Choose inhibitor type ---
# Options: 'competitive', 'noncompetitive', 'uncompetitive', 'mixed', 'none'
MY_INH_TYPE = 'competitive'

# --- Inhibitor concentrations to test ---
MY_I_VALS = [0, 15, 30, 60]   # uM

# --- Inhibitor constants ---
MY_Ki       = 20.0   # uM  (affinity for free E)
MY_Ki_prime = 40.0   # uM  (affinity for ES -- only used for mixed)

# ============================================================
# WRITE YOUR PREDICTION BEFORE RUNNING:
# Inhibition type: MY_INH_TYPE
# I predict Km will:   [ rise / fall / stay same ]
# I predict Vmax will: [ rise / fall / stay same ]
# LB lines will:       [ meet at Y / meet at X / parallel / 2nd quadrant ]
# Ki I will calculate: ~ ___ uM
# ============================================================

S_range_st = np.linspace(0.1, 250, 400)
S_lb_st    = np.array([5, 8, 12, 20, 35, 60, 100, 160])
type_fns = {
    'competitive':    lambda S, I: v_competitive(S, MY_Vmax, MY_Km, I, MY_Ki),
    'noncompetitive': lambda S, I: v_noncompetitive(S, MY_Vmax, MY_Km, I, MY_Ki),
    'uncompetitive':  lambda S, I: v_uncompetitive(S, MY_Vmax, MY_Km, I, MY_Ki),
    'mixed':          lambda S, I: v_mixed(S, MY_Vmax, MY_Km, I, MY_Ki, MY_Ki_prime),
    'none':           lambda S, I: v_noinhib(S, MY_Vmax, MY_Km),
}
fn = type_fns[MY_INH_TYPE]
inh_colors = ['#1D9E75','#7F77DD','#D85A30','#E24B4A']

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle(f'Your Experiment: {MY_INH_TYPE.title()} Inhibition  |  Vmax={MY_Vmax}  Km={MY_Km}  Ki={MY_Ki}',
             fontsize=12, fontweight='bold')

ax1, ax2, ax3 = axes

# v vs [S]
for I, col in zip(MY_I_VALS, inh_colors):
    v = fn(S_range_st, I)
    ax1.plot(S_range_st, v, color=col, label=f'[I]={I} uM')
ax1.set_xlabel('[S] (mM)'); ax1.set_ylabel('v0 (umol/min)')
ax1.set_title('MM curves at each [I]', fontsize=11); ax1.legend(fontsize=9)

# LB plot
for I, col in zip(MY_I_VALS, inh_colors):
    v_lb = fn(S_lb_st, I)
    ax2.plot(1/S_lb_st, 1/v_lb, 'o-', color=col, markersize=4, label=f'[I]={I} uM')
ax2.axvline(0, color='#aaa', lw=0.8); ax2.axhline(0, color='#aaa', lw=0.8)
ax2.set_xlabel('1/[S] (mM-1)'); ax2.set_ylabel('1/v0 (min/umol)')
ax2.set_title('Lineweaver-Burk\nWhere do the lines intersect?', fontsize=11)
ax2.legend(fontsize=9)

# Apparent parameters vs [I]
I_cont = np.linspace(0, max(MY_I_VALS)*1.5, 200)
Km_app_arr, Vm_app_arr = [], []
for I in I_cont:
    v_pts = fn(S_lb_st, I)
    try:
        popt, _ = curve_fit(lambda S,Vm,K: v_noinhib(S,Vm,K), S_lb_st, v_pts,
                             p0=[MY_Vmax, MY_Km], maxfev=3000)
        Vm_app_arr.append(popt[0]); Km_app_arr.append(popt[1])
    except:
        Vm_app_arr.append(np.nan); Km_app_arr.append(np.nan)

ax3.plot(I_cont, Km_app_arr,  color='#7F77DD', linewidth=2.2, label='Apparent Km')
ax3.plot(I_cont, Vm_app_arr,  color='#E24B4A', linewidth=2.2, linestyle='--', label='Apparent Vmax')
ax3.axhline(MY_Km,   color='#7F77DD', linestyle=':', alpha=0.5)
ax3.axhline(MY_Vmax, color='#E24B4A', linestyle=':', alpha=0.5)
ax3.set_xlabel('[I] (uM)'); ax3.set_ylabel('Apparent value')
ax3.set_title('How Km and Vmax change with [I]\nThe slope encodes Ki', fontsize=11)
ax3.legend(fontsize=9)

plt.tight_layout()
plt.show()

print('YOUR RESULTS:')
print(f'  Inhibition type tested: {MY_INH_TYPE}')
print(f'  True Ki set:            {MY_Ki} uM')
print()
print('COMPARE WITH YOUR PREDICTION:')
print('  Expected LB pattern for', MY_INH_TYPE, ':')
if MY_INH_TYPE == 'competitive':
    print('  Lines meet at Y-axis. Slope rises with [I]. Km_app rises. Vmax unchanged.')
    print(f'  Km_app at [I]={MY_I_VALS[-1]}: {MY_Km*(1+MY_I_VALS[-1]/MY_Ki):.1f} mM')
elif MY_INH_TYPE == 'noncompetitive':
    print('  Lines meet at X-axis. Y-intercept rises with [I]. Vmax_app falls. Km unchanged.')
    print(f'  Vmax_app at [I]={MY_I_VALS[-1]}: {MY_Vmax/(1+MY_I_VALS[-1]/MY_Ki):.1f} umol/min')
elif MY_INH_TYPE == 'uncompetitive':
    print('  Parallel lines. Both Km and Vmax fall equally.')
    a = 1+MY_I_VALS[-1]/MY_Ki
    print(f'  At [I]={MY_I_VALS[-1]}: Km_app={MY_Km/a:.1f}  Vmax_app={MY_Vmax/a:.1f}')
elif MY_INH_TYPE == 'mixed':
    print('  Lines meet in 2nd quadrant (left of Y-axis). Vmax falls. Km may rise or fall.')
else:
    print('  No inhibitor -- baseline MM kinetics.')


---
## Summary — The Complete Inhibition Detective Toolkit

| Stage | What you did | What emerged |
|-------|-------------|-------------|
| 1 | Observed anomalously low rate | The concept of an inhibitor as a separate molecular actor |
| 2 | Asked where the inhibitor binds | Four physical situations — three possible binding locations |
| 3 | Tested competitive inhibition | Km rises, Vmax unchanged — substrate can win |
| 4 | Tested non-competitive inhibition | Vmax falls, Km unchanged — substrate cannot help |
| 5 | Tested uncompetitive inhibition | Both fall equally — the paradox of substrate worsening inhibition |
| 6 | Saw the general mixed case | Contains all other types as special cases |
| 7 | Read unknown data | LB intersection pattern as the diagnostic signature |
| 8 | Distinguished reversible from irreversible | Heavy metals — temporary competition vs permanent damage |
| 9 | Measured IC₅₀ | Potency ranking — connecting kinetics to ecotoxicology |
| 10 | Applied to Western Odisha soils | Inhibition type as a contamination fingerprint |
| 11 | Ran your own experiment | Full diagnostic workflow from parameters to identification |

---

### The master diagnostic key

```
OBSERVE: rate is lower than expected
   |
   v
ADD EXCESS SUBSTRATE — does rate recover?
   |
   +-- YES (fully)  -> Competitive    (Km rises, Vmax unchanged, LB meets at Y-axis)
   |
   +-- NO           -> Non-competitive or Mixed
   |                   |
   |                   +-- Km unchanged -> Non-competitive (LB meets at X-axis)
   |                   +-- Km changes   -> Mixed (LB meets in 2nd quadrant)
   |
   +-- WORSE        -> Uncompetitive  (parallel LB lines)

THEN: is inhibition reversible?
   Preincubation test + EDTA/chelator rescue
   Flat activity vs time  -> Reversible
   Falling activity vs time -> Irreversible component present
```

---

> *The inhibition type is not a category to memorise.*  
> *It is a conclusion you reach by asking the right questions of the data.*  
> *That is what it means to be a Pattern Hunter.*

---
*Notebook 2 in the Pattern Hunters Biochemistry series*  
*Kuchinda Degree College | Department of Zoology | Sambalpur University*  
*OSHEC Project: Genomic Responses of Earthworms to Industrial Pollution in Western Odisha*  
*GitHub: https://github.com/The-Pattern-Hunter/Biochemistry*
